# LNG process simulation and benchmark comparison with NeqSim

This notebook compares four closed-loop LNG process templates on a common feed
and reporting basis:

- single mixed refrigerant (SMR);
- propane-precooled mixed refrigerant (C3MR);
- dual mixed refrigerant (DMR); and
- nitrogen expander.

It uses the process API introduced in
[equinor/neqsim PR #2504](https://github.com/equinor/neqsim/pull/2504).
Until that PR is included in a released NeqSim package, this notebook is a
coordinated development example and will stop with a clear version message.


## 1. Install and load NeqSim

After PR #2504 is released, the normal package upgrade is sufficient. Restart
the runtime if Colab asks after the installation.


In [ ]:
%pip -q install --upgrade neqsim pandas matplotlib


In [ ]:
from neqsim import jneqsim
import matplotlib.pyplot as plt
import pandas as pd

try:
    LNGProcessBuilder = jneqsim.process.lng.LNGProcessBuilder
    LNGProcessCycle = jneqsim.process.lng.LNGProcessCycle
    LNGProcessBenchmark = jneqsim.process.lng.LNGProcessBenchmark
    LNGProcessModel = jneqsim.process.lng.LNGProcessModel
except Exception as exc:
    raise RuntimeError(
        "This notebook requires a NeqSim release containing equinor/neqsim PR #2504."
    ) from exc


## 2. Common case definition

A single feed basis makes the route comparison transparent. The built-in feed
is a dry, pretreated natural gas. For a project study, pass a custom
`SystemInterface` with `setFeedFluid` and represent pretreatment upstream.

Six exchanger zones and disabled adaptive refinement keep this first route
screen fast. Increase the zone count and enable refinement for final results.


In [ ]:
FEED_KG_PER_HOUR = 20_000.0
FEED_PRESSURE_BARA = 60.0
NUMBER_OF_ZONES = 6

cycles = {
    "SMR": LNGProcessCycle.SMR,
    "C3MR": LNGProcessCycle.C3MR,
    "DMR": LNGProcessCycle.DMR,
    "Nitrogen expander": LNGProcessCycle.NITROGEN_EXPANDER,
}


## 3. Run all process routes

Every model contains explicit refrigerant recycles and reports the same KPIs:
LNG rate and yield, MTPA, compressor and recovered expander power, specific
energy, product state, minimum exchanger approach, exergy destruction, and
runtime.


In [ ]:
def run_case(label, cycle, zones=NUMBER_OF_ZONES, adaptive=False):
    model = (
        LNGProcessBuilder()
        .setName(f"{label} comparison case")
        .setCycle(cycle)
        .setFeedFlowRate(FEED_KG_PER_HOUR)
        .setFeedPressure(FEED_PRESSURE_BARA)
        .setNumberOfZones(zones)
        .setAdaptiveRefinement(adaptive)
        .build()
    )
    result = model.run()
    assessment = result.assessBenchmark()
    benchmark = LNGProcessBenchmark.get(cycle)
    return {
        "cycle": label,
        "LNG kg/h": result.getLNGMassFlowKgPerHour(),
        "LNG yield": result.getLNGYield(),
        "capacity MTPA": result.getCapacityMTPA(),
        "compressor kW": result.getCompressorPowerKW(),
        "expander kW": result.getExpanderRecoveredPowerKW(),
        "net power kW": result.getNetPowerKW(),
        "SEC kWh/kg LNG": result.getSpecificEnergyKWhPerKgLNG(),
        "literature reference": benchmark.getReferenceSpecificEnergy(),
        "SEC deviation": assessment.getSpecificEnergyDeviation(),
        "product degC": result.getProductTemperatureC(),
        "MITA degC": result.getMinimumInternalTemperatureApproachC(),
        "exchanger exergy kW": result.getExchangerExergyDestructionKW(),
        "runtime ms": result.getRunTimeMilliseconds(),
        "screening pass": assessment.isWithinRange(),
        "diagnostics": "; ".join(str(item) for item in assessment.getMessages()),
    }


rows = [run_case(label, cycle) for label, cycle in cycles.items()]
results = pd.DataFrame(rows).set_index("cycle")
results.round(4)


## 4. Compare specific energy with literature points

The literature markers are comparison points rather than universal acceptance
limits. A strict validation must align feed composition, ambient conditions,
product flash specification, equipment efficiencies, pressure drops, and
flowsheet detail.


In [ ]:
ax = results[["SEC kWh/kg LNG", "literature reference"]].plot(
    kind="bar", figsize=(10, 5), color=["#0072B2", "#D55E00"]
)
ax.set_ylabel("Specific energy [kWh/kg LNG]")
ax.set_xlabel("")
ax.set_title("NeqSim LNG route screen versus literature comparison points")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


Literature basis:

- SMR, C3MR, and DMR: Pereira et al.,
  [Energy Conversion and Management 272 (2022) 116364](https://doi.org/10.1016/j.enconman.2022.116364).
- Parallel nitrogen expansion: He et al.,
  [Energy 167 (2019) 1–12](https://doi.org/10.1016/j.energy.2018.10.169).

The templates are initialized screening flowsheets, not optimized replicas of
the papers. Use the deviation and diagnostic columns to guide calibration and
optimization.


## 5. Exchanger grid-convergence study

A credible result should not depend materially on the exchanger
discretization. The next cell reruns one selected process with successively
finer grids and adaptive refinement enabled.


In [ ]:
selected_label = "C3MR"
selected_cycle = cycles[selected_label]
grid_rows = []

for zones in (4, 8, 16):
    row = run_case(selected_label, selected_cycle, zones=zones, adaptive=True)
    grid_rows.append(
        {
            "zones": zones,
            "SEC kWh/kg LNG": row["SEC kWh/kg LNG"],
            "MITA degC": row["MITA degC"],
            "runtime ms": row["runtime ms"],
        }
    )

grid_study = pd.DataFrame(grid_rows).set_index("zones")
grid_study.round(4)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
grid_study["SEC kWh/kg LNG"].plot(marker="o", ax=axes[0])
axes[0].set_ylabel("Specific energy [kWh/kg LNG]")
axes[0].set_title("Grid convergence")
axes[0].grid(alpha=0.3)

grid_study["runtime ms"].plot(marker="o", ax=axes[1], color="#009E73")
axes[1].set_ylabel("Runtime [ms]")
axes[1].set_title("Accuracy-speed cost")
axes[1].grid(alpha=0.3)

for axis in axes:
    axis.set_xlabel("Base exchanger zones")
plt.tight_layout()
plt.show()


## 6. Build a professional integrated LNG flowsheet and evaluate capacity

A project model normally supplies a treated-gas stream from explicit upstream
equipment. This example creates a rich feed, inlet pipeline, and eight-tray
heavy-hydrocarbon scrub column. The column overhead is passed by reference to
the C3MR train with `setUpstreamProcess`, while the NGL bottoms are published as
an additional named product.

The route builder then adds explicit compressor suction scrubbers, two-stage
compressors, intercoolers and aftercoolers, refrigerant valves, cryogenic
exchangers, product flashing, and closed refrigerant recycles. Liquid LNG is
sent to a transfer pipeline and flash gas to recompression. All units remain in
one `ProcessSystem` for execution, inspection, reporting, mechanical design,
and plant-wide capacity ranking.


In [ ]:
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem
PipeBeggsAndBrills = jneqsim.process.equipment.pipeline.PipeBeggsAndBrills
DistillationColumn = jneqsim.process.equipment.distillation.DistillationColumn
Compressor = jneqsim.process.equipment.compressor.Compressor

rich_feed_fluid = SystemSrkEos(25.0 + 273.15, 70.0)
rich_feed_composition = {
    "nitrogen": 0.010,
    "methane": 0.820,
    "ethane": 0.080,
    "propane": 0.050,
    "n-butane": 0.025,
    "n-pentane": 0.015,
}
for component, fraction in rich_feed_composition.items():
    rich_feed_fluid.addComponent(component, fraction)
rich_feed_fluid.setMixingRule("classic")

plant = ProcessSystem("Professional integrated LNG plant")
rich_feed = Stream("Rich natural-gas feed", rich_feed_fluid)
rich_feed.setFlowRate(FEED_KG_PER_HOUR, "kg/hr")
plant.add(rich_feed)

inlet_pipeline = PipeBeggsAndBrills("Feed-gas pipeline", rich_feed)
inlet_pipeline.setLength(20_000.0)
inlet_pipeline.setDiameter(0.40)
plant.add(inlet_pipeline)

scrub_column = DistillationColumn(
    "Heavy-hydrocarbon scrub column", 8, True, True
)
scrub_column.addFeedStream(inlet_pipeline.getOutletStream(), 4)
scrub_column.setCondenserTemperature(-35.0, "C")
scrub_column.setReboilerTemperature(65.0, "C")
plant.add(scrub_column)

capacity_model = (
    LNGProcessBuilder()
    .setName("Integrated C3MR train")
    .setCycle(LNGProcessCycle.C3MR)
    .setUpstreamProcess(plant, scrub_column.getGasOutStream())
    .setNumberOfZones(NUMBER_OF_ZONES)
    .setAdaptiveRefinement(False)
    .build()
)
capacity_model.registerOutputStream("NGL", scrub_column.getLiquidOutStream())

equipment_inventory = pd.DataFrame(
    [
        {
            "name": str(unit.getName()),
            "unit operation": str(unit.getClass().getSimpleName()),
        }
        for unit in capacity_model.getEquipment()
    ]
).set_index("name")
equipment_inventory


In [ ]:
lng_stream = capacity_model.getOutputStream(LNGProcessModel.LNG_OUTPUT)
flash_gas_stream = capacity_model.getOutputStream(
    LNGProcessModel.FLASH_GAS_OUTPUT
)
ngl_stream = capacity_model.getOutputStream("NGL")

product_pipeline = PipeBeggsAndBrills("LNG product pipeline", lng_stream)
product_pipeline.setLength(15_000.0)
product_pipeline.setDiameter(0.50)
capacity_model.addEquipment(product_pipeline)

flash_gas_compressor = Compressor("LNG flash gas compressor", flash_gas_stream)
flash_gas_compressor.setOutletPressure(10.0, "bara")
flash_gas_compressor.setIsentropicEfficiency(0.75)
capacity_model.addEquipment(flash_gas_compressor)

capacity = capacity_model.autoSizeAndEvaluateCapacity(1.20)
ranked_capacity = {
    str(entry.getKey()): float(entry.getValue())
    for entry in capacity.getRankedUtilizationPercent().entrySet()
}
capacity_table = pd.DataFrame(
    ranked_capacity.items(), columns=["equipment", "utilization percent"]
).set_index("equipment")
capacity_table


In [ ]:
bottleneck = capacity.getBottleneck()
capacity_summary = {
    "bottleneck equipment": str(bottleneck.getEquipmentName()),
    "limiting constraint": str(bottleneck.getConstraintName()),
    "utilization percent": float(bottleneck.getUtilizationPercent()),
    "any equipment overloaded": bool(capacity.isAnyEquipmentOverloaded()),
    "any hard limit exceeded": bool(capacity.isAnyHardLimitExceeded()),
    "auto-sized equipment": int(capacity.getAutoSizedEquipmentCount()),
    "derived constraints": int(capacity.getDerivedConstraintCount()),
}
capacity_summary


For an existing facility, configure the actual mechanical-design envelopes
and call `evaluateCapacity()` after running instead of auto-sizing from the
current operating point. The full per-constraint payload is available from
`capacity.getUtilizationSnapshotJson()`, and the sizing basis from
`capacity.getDesignReportJson()`.

A maximum-throughput study should vary the live feed stream rate with NeqSim's
optimization framework until the first hard or design constraint binds.


## 7. Recommended next steps

For engineering work, replace the representative scrub column with the actual
pretreatment and NGL-recovery scheme, then calibrate refrigerant compositions,
circulation rates, pressure levels, exchanger pressure drops, and rotating
equipment maps. For detailed C3MR design, expand the equivalent single-level
propane precooler into the actual multi-level cascade. Compare against plant or
licensor data only after matching the complete case definition.
